<a href="https://colab.research.google.com/github/Oumohgh/ChestScan_AI-/blob/main/Brief_transfer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import les biblio

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import kagglehub
from google.colab import userdata
import tensorflow as tf
import os
import pprint as pp
from PIL import Image, ImageOps
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import random
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve, ConfusionMatrixDisplay)

# EDA

In [2]:
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print(path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
/kaggle/input/chest-xray-pneumonia


In [3]:
!ls {path}/chest_xray/train/NORMAL | head -n 10

IM-0115-0001.jpeg
IM-0117-0001.jpeg
IM-0119-0001.jpeg
IM-0122-0001.jpeg
IM-0125-0001.jpeg
IM-0127-0001.jpeg
IM-0128-0001.jpeg
IM-0129-0001.jpeg
IM-0131-0001.jpeg
IM-0133-0001.jpeg


## Retrieval of raw images

In [4]:

training_path = path + "/chest_xray/train"
testing_path = path + "/chest_xray/test"
validation_path = path + "/chest_xray/val"

In [5]:
allowed_extensions = ("png", "jpg", "jpeg")
def get_images(new_path: str):
  dataset = tf.keras.preprocessing.image_dataset_from_directory(new_path, label_mode='binary', batch_size=None, color_mode="rgb", image_size=(180,180), crop_to_aspect_ratio=False, shuffle=True)



  return dataset


In [6]:
# Dataframe with two columns image & label, image: Image refrence, label: normal || pneumonia
raw_training_images = get_images(training_path)
# pp.pprint(raw_training_images)


Found 5216 files belonging to 2 classes.


In [7]:
def convert_to_panadas(dataset):
  ds_labels=[]
  for image, labels in dataset:
     ds_labels.append({"image": image.numpy(), "label": labels.numpy()[0]})
  new_df = pd.DataFrame.from_dict(ds_labels)
  return new_df


In [8]:
raw_training_images = convert_to_panadas(raw_training_images)

In [9]:
raw_testing_images = get_images(testing_path)

Found 624 files belonging to 2 classes.


In [ ]:
raw_testing_images = convert_to_panadas(raw_testing_images)

In [ ]:
raw_validation_images = get_images(validation_path)

In [ ]:
raw_validation_images = convert_to_panadas(raw_validation_images)

## Data analysis

In [ ]:
plt.figure(figsize=(10, 10))
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 4))

sns.countplot(x="label", data=raw_training_images, ax=axes[0])
axes[0].set_title("Training images")

sns.countplot(x="label", data=raw_testing_images, ax=axes[1])
axes[1].set_title("Testing images")

sns.countplot(x="label", data=raw_validation_images, ax=axes[2])
axes[2].set_title("Validation images")

plt.show()

In [ ]:
training_count = raw_training_images.groupby("label").count()
testing_count = raw_testing_images.groupby("label").count()
validation_count = raw_validation_images.groupby("label").count()

stats = {
    "training": {
        "total": len(raw_training_images),
        "normal_percentage": (training_count.loc[0, "image"] * 100) / len(raw_training_images),
        "pneumonia_percentage": (training_count.loc[1, "image"] * 100) / len(raw_training_images)
    },
    "testing": {
        "total": len(raw_testing_images),
        "normal_percentage": (testing_count.loc[0, "image"] * 100) / len(raw_testing_images),
        "pneumonia_percentage": (testing_count.loc[1, "image"] * 100) / len(raw_testing_images)
    },
    "validation": {
        "total": len(raw_validation_images),
        "normal_percentage": (validation_count.loc[0, "image"] * 100) / len(raw_validation_images),
        "pneumonia_percentage": (validation_count.loc[1, "image"] * 100) / len(raw_validation_images)
    }
}

stats_df = pd.DataFrame(stats)
stats_df

## Image resizing & sanity check

In [ ]:



def scale_vector(vecotre):
  return vecotre / 255.0

In [ ]:
scaled_training_images = raw_training_images
scaled_training_images["image"] = scaled_training_images["image"].map(scale_vector)
scaled_training_images.dropna(inplace=True)

In [ ]:
scaled_testing_images = raw_testing_images.copy()
scaled_testing_images["image"] = scaled_testing_images["image"].map(scale_vector)
scaled_testing_images.dropna(inplace=True)

In [ ]:
scaled_validation_images = raw_validation_images
scaled_validation_images["image"] = scaled_validation_images["image"].map(scale_vector)
scaled_validation_images.dropna(inplace=True)

In [ ]:
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(scaled_training_images.iloc[i]["image"])
    plt.xlabel(scaled_training_images.iloc[i]['label'])
plt.show()

In [ ]:
def get_weights(df: pd.DataFrame):
  labels = df["label"].unique()
  weights = compute_class_weight(class_weight="balanced", classes=labels, y=df["label"])
  class_weights = dict(zip(labels, weights))
  return class_weights

# Modelisation

In [ ]:
model = models.Sequential([
    layers.Input(shape=(180, 180, 3)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.Flatten(),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

In [ ]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy", "precision", "recall", "auc"])

X_train = np.stack(scaled_training_images['image'].to_list()).astype("float32")
y_train = scaled_training_images['label'].to_numpy().astype("float32")

x_val = np.stack(scaled_validation_images['image'].to_list()).astype("float32")
y_val = scaled_validation_images['label'].to_numpy().astype("float32")

class_weights = {int(k): float(v) for k, v in get_weights(scaled_training_images).items()}

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc", mode="max", patience=5, restore_best_weights=True
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
)

history = model.fit(
    X_train, y_train,
    batch_size=64,
    epochs=10,
    class_weight=class_weights,
    validation_data=(x_val, y_val),
    callbacks=[early_stop, reduce_lr]
)



In [ ]:
h = history.history
epochs = range(1, len(h["loss"]) + 1)
best_epoch = int(np.argmax(h["val_auc"])) + 1


lr = h["learning_rate"]
lr_drops = [i + 1 for i in range(1, len(lr)) if lr[i] < lr[i - 1]]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for ax, (metric, title) in zip(axes[:2], [("loss", "Loss"), ("auc", "AUC")]):
    ax.plot(epochs, h[metric], label="train")
    ax.plot(epochs, h[f"val_{metric}"], label="validation")
    ax.axvline(best_epoch, color="green", ls="--", label=f"best epoch ({best_epoch})")
    for e in lr_drops:
        ax.axvline(e, color="orange", ls=":", alpha=0.8)
    ax.set_title(title); ax.set_xlabel("epoch"); ax.legend()

axes[2].plot(epochs, h["recall"], label="train recall")
axes[2].plot(epochs, h["val_recall"], label="val recall")
axes[2].plot(epochs, h["val_precision"], label="val precision")
axes[2].axvline(best_epoch, color="green", ls="--")
axes[2].set_title("Recall / Precision"); axes[2].set_xlabel("epoch"); axes[2].legend()

plt.tight_layout(); plt.show()

print("Stopped at epoch:", len(epochs), "| best epoch:", best_epoch)
print("LR reduced at epochs:", lr_drops)
print(f"Final gap  loss: {h['val_loss'][-1] - h['loss'][-1]:.3f} | "
      f"auc: {h['auc'][-1] - h['val_auc'][-1]:.3f}")

In [ ]:
X_test = np.stack(scaled_testing_images['image'].to_list()).astype("float32")
y_test = scaled_testing_images['label'].to_numpy().astype("int")


In [ ]:
THRESHOLD = 0.5

def evaluate(X, y, threshold=THRESHOLD):
    proba = model.predict(X, batch_size=64, verbose=0).ravel()
    pred = (proba >= threshold).astype(int)
    return {
        "Accuracy":  accuracy_score(y, pred),
        "Precision": precision_score(y, pred),
        "Recall":    recall_score(y, pred),
        "F1":        f1_score(y, pred),
        "ROC-AUC":   roc_auc_score(y, proba),
    }, proba, pred

test_metrics, test_proba, test_pred = evaluate(X_test, y_test)
print(test_metrics)
print(classification_report(y_test, test_pred, target_names=["NORMAL", "PNEUMONIA"]))
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, display_labels=["NORMAL", "PNEUMONIA"]); plt.show()

In [ ]:
y_tr_int  = y_train.astype(int)       #
y_val_int = y_val.astype(int)

train_metrics, _, _ = evaluate(X_train, y_tr_int)
val_metrics, _, _   = evaluate(x_val, y_val_int)

results = pd.DataFrame({"Train": train_metrics,
                        "Validation": val_metrics,
                        "Test": test_metrics}).T.round(3)
results

In [ ]:
rows = []
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    p = (test_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    rows.append({"threshold": t,
                 "precision": precision_score(y_test, p, zero_division=0),
                 "recall": recall_score(y_test, p),
                 "specificity": tn / (tn + fp),
                 "FN (missed pneumonia)": fn,
                 "FP (false alarms)": fp})
pd.DataFrame(rows).round(3)

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_proba)
plt.plot(fpr, tpr, label=f"AUC = {test_metrics['ROC-AUC']:.3f}")
plt.plot([0, 1], [0, 1], "k--", label="random")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate (recall)")
plt.legend(); plt.show()

In [ ]:
class_names = ["NORMAL", "PNEUMONIA"]

correct = np.where(test_pred == y_test)[0]
fp_idx = np.where((test_pred == 1) & (y_test == 0))[0]
fn_idx = np.where((test_pred == 0) & (y_test == 1))[0]

print(f"Correct: {len(correct)} | FP: {len(fp_idx)} | FN: {len(fn_idx)}")

In [ ]:
def show_grid(indices, title, n=12, cols=4, seed=42):
    rng = np.random.default_rng(seed)
    if len(indices) == 0:
        print(f"No images for: {title}")
        return
    chosen = rng.choice(indices, size=min(n, len(indices)), replace=False)
    rows = int(np.ceil(len(chosen) / cols))
    plt.figure(figsize=(cols * 3.2, rows * 3.4))
    for k, i in enumerate(chosen):
        plt.subplot(rows, cols, k + 1)
        plt.imshow(X_test[i])
        plt.axis("off")
        plt.title(f"true: {class_names[int(y_test[i])]}\n"
                  f"score: {test_proba[i]:.2f}", fontsize=9,
                  color="green" if test_pred[i] == y_test[i] else "red")
    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:

show_grid(correct[y_test[correct] == 1], "Correctly classified: PNEUMONIA", n=8)
show_grid(correct[y_test[correct] == 0], "Correctly classified: NORMAL", n=8)


show_grid(fp_idx, "False positives (NORMAL predicted as PNEUMONIA)", n=8)
show_grid(fn_idx, "False negatives (PNEUMONIA predicted as NORMAL)", n=8)

In [ ]:
worst_fp = fp_idx[np.argsort(-test_proba[fp_idx])][:8]
worst_fn = fn_idx[np.argsort(test_proba[fn_idx])][:8]
show_grid(worst_fp, "Most confident false positives", n=8)
show_grid(worst_fn, "Most confident false negatives", n=8)

In [ ]:
def stats(idx):
    imgs = X_test[idx]
    return {"n": len(idx),
            "mean brightness": imgs.mean(),
            "contrast (std)": imgs.std(axis=(1, 2, 3)).mean()}

pattern_df = pd.DataFrame({
    "Correct": stats(correct),
    "False positives": stats(fp_idx),
    "False negatives": stats(fn_idx),
}).T.round(3)
pattern_df


plt.hist(test_proba[y_test == 0], bins=30, alpha=0.6, label="true NORMAL")
plt.hist(test_proba[y_test == 1], bins=30, alpha=0.6, label="true PNEUMONIA")
plt.axvline(THRESHOLD, color="k", ls="--", label=f"threshold {THRESHOLD}")
plt.xlabel("predicted probability of PNEUMONIA"); plt.legend(); plt.show()

In [ ]:
cnn_model = model
cnn_model.save("cnn_model.keras")

print("CNN initial sauvegardé avec succès.")


# Transfer learninig

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow import data as tf_sdata

In [ ]:
resize_fn = keras.layers.Resizing(180,180)

In [ ]:
augmentation_layers = {
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1)
}

def data_augmentation(x):
  for layers in augmentation_layers:
    x = layers(x)
  return x

In [ ]:
base_model = MobileNetV2(
    input_shape=(180, 180, 3),
    weights="imagenet",
    include_top=False
)
base_model.trainable = False

#  for layer in base_model.layers:
#    print(layer)

In [ ]:
inputs = keras.Input(shape=(180, 180, 3))
scale_layer = keras.layers.Rescaling(scale=1 / 127.5, offset=-1)
x = scale_layer(inputs)
x = base_model(x, training=False)

In [ ]:
x = keras.layers.GlobalAveragePooling2D()(x)

In [ ]:
x = layers.Dropout(0.2)(x)

In [ ]:
outputs = keras.layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

In [ ]:
model.compile(
    optimizer= keras.optimizers.Adam(),
    loss= keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=["accuracy", "precision", "recall", "auc"]
)


In [ ]:
epochs = 5
print("Fitting the top layer of the model")
ransfer_learning_history = model.fit(
    X_train, y_train,
    epochs=epochs,
    class_weight=class_weights,
    validation_data=(x_val, y_val),
    callbacks=[early_stop, reduce_lr]
    )

In [ ]:
tl_model = model
tl_model.save("mobilenetv2_transfer.keras")

print("Modèle Transfer Learning sauvegardé avec succes.")

# Fine tunning

In [ ]:
len(base_model.layers)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:90]:
  layer.trainable = False

model.compile(
    optimizer= keras.optimizers.Adam(0.00001),
    loss= keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=["accuracy", "precision", "recall", "auc"]
)


In [ ]:
fine_tuned_history = model.fit(
    X_train, y_train,
    epochs=epochs,
    class_weight=class_weights,
    validation_data=(x_val, y_val),
    callbacks=[early_stop, reduce_lr]
    )

In [ ]:
transfer_learning_history = ransfer_learning_history.history
fine_tuned_history = fine_tuned_history.history

In [ ]:
transfer_learning_history

In [ ]:
base_model.summary()

In [ ]:

ft_model = model
ft_model.save("mobilenetv2_finetuned.keras")

print("Modèle Fine-Tuning sauvegardé avec succès.")

In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from tensorflow.keras.models import load_model

SEUIL = 0.5
CLASSE_POSITIVE = 1  # 1 = PNEUMONIA
NOMS_MODELES = ["1. CNN", "2. Transfer Learning", "3. Fine-Tuning"]
METRIQUES = ["accuracy", "precision", "recall", "F1", "ROC-AUC", "loss"]


CANDIDATS = {
    "1. CNN": ("cnn_model", ["cnn_model.keras", "cnn_scratch.keras"]),
    "2. Transfer Learning": ("tl_model", ["mobilenetv2_transfer.keras", "tl_model.keras"]),
    "3. Fine-Tuning": ("ft_model", ["mobilenetv2_finetuned.keras", "chestscan_model.keras"]),
}

modeles, origines = {}, {}
for nom in NOMS_MODELES:
    variable, fichiers = CANDIDATS[nom]

    if variable in globals():
        modeles[nom] = globals()[variable]
        origines[nom] = f"variable `{variable}`"
    else:
        fichier = next((f for f in fichiers if os.path.isfile(f)), None)
        if fichier:
            modeles[nom] = load_model(fichier)
            origines[nom] = f"fichier `{fichier}`"
        else:
            modeles[nom] = None
            origines[nom] = "indisponible"


if modeles["3. Fine-Tuning"] is None and "model" in globals():
    modeles["3. Fine-Tuning"] = model
    origines["3. Fine-Tuning"] = "variable `model` (modèle après fine-tuning)"

print("Modèles utilisés :")
for nom in NOMS_MODELES:
    print(f"  {nom:<24} -> {origines[nom]}")

if modeles["1. CNN"] is None:
    print("\nCNN non disponible : le CNN initial a été remplacé par `model`.")
    print("Pour le comparer, sauvegardez-le avant de réutiliser `model`,")
    print("puis chargez-le sous le nom `cnn_model` ou dans un fichier .keras.")
if modeles["2. Transfer Learning"] is None:
    print("\nModèle Transfer Learning non disponible séparément.")
    print("Il faut le sauvegarder après son entraînement et avant le fine-tuning.")

def calculer_metriques(modele, X, y, seuil=SEUIL):
    """Calcule les métriques sur le même jeu de test pour chaque modèle."""
    sorties = np.asarray(modele.predict(X, batch_size=64, verbose=0)).reshape(-1)

    perte = getattr(modele, "loss", None)
    from_logits = bool(getattr(perte, "from_logits", False))

    if from_logits:
        probabilites = tf.math.sigmoid(sorties).numpy()
    else:
        probabilites = sorties

    predictions = (probabilites >= seuil).astype(int)
    y_int = np.asarray(y).astype(int).reshape(-1)


    auc = roc_auc_score(y_int, probabilites) if len(np.unique(y_int)) == 2 else np.nan
    perte_test = modele.evaluate(X, y, batch_size=64, verbose=0, return_dict=True)["loss"]

    return {
        "accuracy": accuracy_score(y_int, predictions),
        "precision": precision_score(y_int, predictions, pos_label=CLASSE_POSITIVE, zero_division=0),
        "recall": recall_score(y_int, predictions, pos_label=CLASSE_POSITIVE, zero_division=0),
        "F1": f1_score(y_int, predictions, pos_label=CLASSE_POSITIVE, zero_division=0),
        "ROC-AUC": auc,
        "loss": perte_test,
    }


resultats = {}
for nom in NOMS_MODELES:
    if modeles[nom] is None:
        resultats[nom] = {m: np.nan for m in METRIQUES}
    else:
        resultats[nom] = calculer_metriques(modeles[nom], X_test, y_test)

table_comparaison = pd.DataFrame(resultats).reindex(METRIQUES)
display(table_comparaison.style.format("{:.3f}", na_rep="—"))


metriques_graphique = ["precision", "recall", "F1", "ROC-AUC"]
ax = table_comparaison.loc[metriques_graphique].T.plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1.05)
)
ax.set_title("Comparaison des modèles sur le jeu de test")
ax.set_xlabel("Modèle")
ax.set_ylabel(f"Score (seuil = {SEUIL})")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

def historique_en_dict(obj):
    if isinstance(obj, dict):
        return obj
    return getattr(obj, "history", None) if obj is not None else None

historiques = [
    ("1. CNN", globals().get("history")),
    ("2. Transfer Learning", globals().get("ransfer_learning_history")),
    ("3. Fine-Tuning", globals().get("fine_tuned_history")),
]
plt.figure(figsize=(8, 4.5))
nb_courbes = 0
for nom, obj in historiques:
    hist = historique_en_dict(obj)
    if hist and "val_loss" in hist:
        plt.plot(range(1, len(hist["val_loss"]) + 1), hist["val_loss"], marker="o", label=nom)
        nb_courbes += 1
if nb_courbes:
    plt.xlabel("Époque")
    plt.ylabel("Validation loss")
    plt.title("Évolution de la perte de validation")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    plt.close()


recalls = table_comparaison.loc["recall"].dropna()
aucs = table_comparaison.loc["ROC-AUC"].dropna()
if not recalls.empty:
    print(f"Meilleur recall PNEUMONIA : {recalls.idxmax()} ({recalls.max():.3f})")
if not aucs.empty:
    print(f"Meilleur ROC-AUC : {aucs.idxmax()} ({aucs.max():.3f})")
if len(recalls) < len(NOMS_MODELES):
    print(f"\nAttention : {len(recalls)}/{len(NOMS_MODELES)} modèles sont disponibles pour la comparaison.")
    print("Une comparaison équitable nécessite les trois modèles sauvegardés séparément.")